# Module 1: Generative AI Foundations
**Beginner | Suggested schedule: Day 1–Day 5**

This notebook covers Generative AI concepts, LLM fundamentals, Transformers, model types, hallucinations, bias, responsible AI, environment setup, and hands-on model/prompt experiments.

**Learning outcomes**
- Explain tokens, context windows, LLMs, and Transformers.
- Recognize common model families and their limitations.
- Apply responsible AI practices.
- Compare model outputs and prompts in a notebook.

Run cells from top to bottom. The hands-on section downloads small public models and requires internet access. No paid API key is needed.

## 1. Generative AI basics

**AI** is the broad field of systems performing tasks associated with intelligence. **Machine Learning** learns patterns from data. **Deep Learning** uses multi-layer neural networks. **Generative AI** creates new content such as text, images, audio, video, or code.

**Applications:** chatbots, summarization, translation, writing, code assistance, image generation, transcription, and document Q&A.

A model generates outputs based on learned patterns. Fluent wording does not guarantee that an answer is true, current, unbiased, or suitable for a high-stakes decision.

## 2. LLM fundamentals

A **Large Language Model (LLM)** learns patterns in language; many text LLMs generate answers by predicting the next token repeatedly.

- **Token:** a unit of text processed by a model. It can be a word, part of a word, punctuation, or another unit.
- **Tokenization:** converting text into tokens/token IDs.
- **Context window:** the amount of input and generated content a model can process in one interaction.
- **Parameters:** learned numerical values in a model.
- **Inference:** running a trained model to generate an output.
- **Prompt:** instructions and context given to a model.
- **Temperature:** a generation setting that can affect randomness.

A token is not always one word; token counts depend on the tokenizer.

## 3. Transformer architecture

Many modern LLMs use the **Transformer** architecture:
1. Split text into tokens.
2. Convert tokens into numerical representations (embeddings).
3. Use **attention** to represent relationships between tokens in context.
4. Process representations through Transformer layers.
5. Produce scores for possible next tokens.
6. Decode selected tokens into output.

Attention helps a model use context, but it does not guarantee human-like understanding or factual correctness.

**Typical GenAI application:** User interface → input validation → model/API → output checks/guardrails → response → logging/feedback. Protect API keys, handle errors, and avoid sending unnecessary personal data.

## 4. Common model families

| Family | Common uses | Limitations to consider |
|---|---|---|
| Autoregressive text models | Chat, writing, code | May hallucinate; prompt-sensitive |
| Encoder-only models | Classification, semantic search | Usually not intended for open-ended generation |
| Encoder-decoder models | Summarization, translation, instruction tasks | Task and training dependent |
| Diffusion models | Image/media generation | May create artifacts or misleading details |
| Multimodal models | Text + image/audio/video tasks | May misinterpret inputs or miss context |

Choose a model based on task quality, latency, cost, privacy, licensing, and safety—not popularity alone.

## 5. Hallucinations, bias, and responsible AI

**Hallucination:** an output that is false, unsupported, or fabricated, sometimes stated confidently. Verify important claims with reliable sources.

**Bias:** unfair or stereotyped outputs can arise from unbalanced training data, historical inequalities, annotation choices, or system design. Test across different groups and contexts.

Other limitations include outdated knowledge, ambiguous instructions, incorrect code/advice, privacy risks, and sensitivity to prompt wording.

**Responsible AI checklist**
- Verify important facts.
- Never paste passwords, API keys, or unnecessary private data into prompts.
- Test for bias and harmful outputs.
- Disclose AI use when appropriate.
- Validate outputs and protect credentials.
- Keep human oversight for consequential decisions.
- Follow copyright and model-license terms.

## 6. Environment setup

This notebook runs in Google Colab or Jupyter. The next cell installs the open-source libraries used for the demonstration. Public models are downloaded from the Hugging Face model hub; first-time downloads may take a few minutes.

In [ ]:
%pip -q install transformers torch

## 7. Experiment A — tokenization

Inspect how a model's tokenizer splits a sentence. Results vary by model.

In [ ]:
from transformers import AutoTokenizer
tokenizer_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
sample = "Explain Generative AI to a first-year engineering student in three simple points."
pieces = tokenizer.tokenize(sample)
ids = tokenizer.encode(sample, add_special_tokens=True)
print("Text:", sample)
print("Whitespace word count:", len(sample.split()))
print("Token ID count (including special tokens):", len(ids))
print("Token pieces:", pieces)
print("Tokenizer:", tokenizer_name)

## 8. Experiment B — compare two models

We will compare:
- **FLAN-T5-small:** instruction-tuned text-to-text model.
- **DistilGPT-2:** smaller causal language model that continues text and is not instruction-tuned in the same way.

This is a small educational experiment, not a universal model ranking. Their training objectives differ.

In [ ]:
from transformers import pipeline, set_seed
import torch
device = 0 if torch.cuda.is_available() else -1
print("Using:", "GPU" if device == 0 else "CPU")
flan = pipeline("text2text-generation", model="google/flan-t5-small", device=device)
gpt2 = pipeline("text-generation", model="distilgpt2", device=device)
set_seed(42)
print("Models loaded.")

### Experiment 1 — same topic, different model behavior

Run this and compare relevance, clarity, structure, and instruction-following. DistilGPT-2 may continue the prompt instead of following the instruction directly.

In [ ]:
prompt = "Explain Generative AI in three simple bullet points for a beginner."
a = flan(prompt, max_new_tokens=100, do_sample=False)[0]["generated_text"]
b = gpt2(prompt, max_new_tokens=100, do_sample=False,
         pad_token_id=gpt2.tokenizer.eos_token_id)[0]["generated_text"]
print("=== FLAN-T5-small ===\n", a)
print("\n=== DistilGPT-2 ===\n", b)

### Experiment 2 — prompt engineering

A good prompt often specifies audience, task, context, constraints, and output format. Compare the vague and specific prompts below.

In [ ]:
vague = "Tell me about AI."
specific = (
    "You are a patient computer science tutor. Explain Generative AI to a beginner "
    "in exactly 3 numbered points. Use simple English, give one everyday example, "
    "and do not claim AI answers are always correct."
)
print("=== Vague prompt ===")
print(flan(vague, max_new_tokens=100, do_sample=False)[0]["generated_text"])
print("\n=== Specific prompt ===")
print(flan(specific, max_new_tokens=140, do_sample=False)[0]["generated_text"])

### Experiment 3 — structured output

A prompt asking for a format does not guarantee perfect compliance. Inspect the result.

In [ ]:
task = ("Compare rule-based software and Generative AI for a beginner. "
        "Return exactly three bullet points, each containing one difference.")
print(flan(task, max_new_tokens=120, do_sample=False)[0]["generated_text"])

## 9. Observation table

Complete this table using the outputs you actually saw.

| Experiment | Observation | Best fit for task? | Evidence |
|---|---|---|---|
| FLAN-T5 vs DistilGPT-2 | Fill in | Fill in | Instruction-following, relevance, clarity |
| Vague vs specific prompt | Fill in | Fill in | Structure and level of detail |
| Structured output | Fill in | Fill in | Did it follow the requested format? |

Optional scoring: rate relevance, clarity, instruction-following, format, and factual reliability from 1 (poor) to 5 (excellent). Verify facts independently; polished language is not proof.

## 10. Knowledge check

1. What is Generative AI, and how does it differ from rule-based software?
2. What is a token? Why is it not always a word?
3. What does a context window limit?
4. What does attention do in a Transformer?
5. What is a hallucination, and how can you reduce its impact?
6. Name two possible sources of AI bias.
7. Why should API keys and private data be kept out of prompts?
8. What did you observe when comparing the two models?
9. How did prompt specificity affect the output?
10. When should a human verify AI output?

**Short answer guide:** GenAI creates content from learned patterns; tokens are tokenizer-specific units; a context window limits input/output length; attention represents token relationships; hallucinations need fact-checking; bias can arise from data or design; protect secrets; answer experiment questions using your own outputs.

## 11. Submission checklist

- [ ] Run the notebook from top to bottom in Google Colab.
- [ ] Keep outputs visible and complete the observation table.
- [ ] Save a copy in Google Drive.
- [ ] Set notebook sharing to viewer access according to your program's policy.
- [ ] Copy the learning notes into Google Docs or Notion and share the page.
- [ ] Publish the LinkedIn post and copy its URL.
- [ ] Submit the Colab link and learning-notes link in the form.

A shareable Colab/Docs/LinkedIn URL can only be created from your own accounts after uploading or publishing.